In [63]:
from dotenv import load_dotenv
import os

load_dotenv()

import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.documents import Document

from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

index = faiss.IndexFlatL2(1536)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

with open("SchemaText.txt", "r") as f:
    text = f.read()

# optionally split by table
chunks = text.strip().split("\n\n")  # one per table
vector_store.add_texts(chunks)
#vector_store.add_texts(["Trying texttosqlagent","Trying sqlagent"])

['fa880100-1fd4-4a39-8446-9603e611f4fd',
 '9b721a3b-4e98-4a44-b5b6-67df2be9ae17',
 'a7a3a858-da24-45a2-848b-0fc6b9d47efb',
 '7493f503-327c-4546-84f6-81a4fcedeb83',
 'da7352e1-9405-49d2-90e1-fe6113300143',
 'a03f3ff9-c582-43b0-82f0-d5d8717c9ae8',
 'c8ad5c70-f132-4990-9c3d-9aff883d072f',
 'd0ee2af8-6c1f-499c-bd29-694aaff13b79',
 '554e8e61-fda9-40ad-9284-4fbb69d4cf10',
 '6cafe877-e481-4154-b6bc-753d9dd4a13b',
 'fe68df2e-9986-429f-a2ba-4a5ab2087f82',
 'bf51c1eb-fdf7-42dd-9650-85c2716bd3e4',
 '72825d9f-7649-4490-9b5e-735f2766926e',
 'f9bbf934-4fc4-4c83-a43c-0e7d04332b4b',
 '9f1b7841-bbec-41c4-a618-ec5d89cf1d0d',
 '45fe9033-17bd-4109-94e6-1be60f993ad7',
 '9ca0e957-5402-4f81-9bc9-a0460bdd77cc',
 'aa186d67-7547-4d86-8807-32e69a5d587e',
 'd11f0b1a-b45a-425d-84e6-4ff2459a4116',
 'e8ddb83f-edd6-441e-baae-8261ebe5ed33',
 '8bb83420-6eb6-44ff-964d-cdfe28b70588',
 'eabf846b-7707-499c-9d78-3871f185222f',
 'f09f5006-6bcd-4e4a-9331-d08fd75d5c1f',
 '9889cee9-1f99-4f05-923b-916bb44ff3aa',
 '13edab43-ec30-

In [64]:
from langchain_community.document_loaders import TextLoader
filepath="SchemaText.txt"
loader=TextLoader(filepath)
print(len(loader.load()))
pages=loader.load()


1


In [65]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,#hyperparameter
    chunk_overlap=50 #hyperparemeter
)

split_docs = splitter.split_documents(pages)
len(split_docs)
print(split_docs)
index=faiss.IndexFlatIP(384)

retriever=vector_store.as_retriever(
    search_kwargs={"k": 10} #hyperparameter
)

[Document(metadata={'source': 'SchemaText.txt'}, page_content='Table: CUSTOMER\nDescription: Stores information about customers who are eligible for offers or have made purchases. Each customer has a unique ID and basic profile data.\n\ncustomer_id: Unique identifier for the customer (UUID).\n\nfull_name: Full name of the customer.\n\nemail: Email address of the customer. Must be unique.\n\nphone_number: Optional phone number for contacting the customer.\n\naddress: Full physical mailing address of the customer.'), Document(metadata={'source': 'SchemaText.txt'}, page_content='customer_segment: Category or classification of the customer (e.g., Retail, Wholesale, VIP).\n\ncreated_at: Timestamp indicating when the customer record was created.\n\nupdated_at: Timestamp indicating when the customer record was last modified.\n\n\nTable: PRODUCT\nDescription: Contains information about products that can have offers applied to them. Each product is uniquely identified and categorized.\n\nproduc

In [66]:
from dotenv import load_dotenv
import os

load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model='gemini-1.5-flash')

In [67]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.chat import SystemMessagePromptTemplate, HumanMessagePromptTemplate

# Define prompt template
sql_chat_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a helpful AI assistant that generates SQL queries based on user questions and a database schema."
    ),
    HumanMessagePromptTemplate.from_template("""
You will receive:
1. A database schema
2. A user question

Use the schema to generate a valid SQL query that answers the question.

### Format Instructions:
- Only return the SQL query. Do NOT include explanations or preambles.
- Format the SQL using proper line breaks and indentation.
- Use table aliases where appropriate.
- Use ANSI SQL syntax.
- If the schema is insufficient to answer the question, respond with: `-- Cannot answer with the given schema.`

### Schema:
{context}

### Question:
{question}

### SQL:
""")
])

print(sql_chat_prompt.input_variables)

['context', 'question']


In [68]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

import pprint
pprint.pprint(prompt.messages)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda
#context(retriever),prompt(hub),model(google),parser(langchain)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | sql_chat_prompt
    #| RunnableLambda(lambda x: x.to_messages())
    | model
    | StrOutputParser()
)

#generatedresponse = rag_chain.invoke("find the customer with offers and products for customer prakashbabu for product creditcard")
#response = rag_chain.invoke({"question": "how many customers are available with active offers hat are loaded today"})
response = rag_chain.invoke( "how many customers are available with active offers hat are loaded today")

print(response)

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]
```sql
SELECT
  COUNT(DISTINCT c.customer_id)
FROM CUSTOMER AS c
JOIN OFFER AS o
  ON c.customer_id = o.customer_id
WHERE
  o.status = 'Active' AND o.effective_date = CURRENT_DATE;
```


In [69]:
import os
db_path = "my_database.db"
if os.path.exists(db_path):
    print(f"Database exists at: {os.path.abspath(db_path)}")
else:
    print("Database will be created")

Database exists at: /Users/prakashbabupolisey/Prakash/GenAI/SQLAgent/Demo/my_database.db


In [70]:
import os
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_sql_agent
from langchain_community.utilities import SQLDatabase
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_openai import ChatOpenAI  # Uncomment if using OpenAI

# ✅ Load .env with DB path and API key
from dotenv import load_dotenv
load_dotenv()

# ✅ Load the SQLite database
db = SQLDatabase.from_uri("sqlite:///my_database.db")  # Adjust path
print(f"Database tables: {db.get_table_names()}")
#print(f"Sample rows from Customer: {db.get_sample_rows('Customer')}")

# ✅ Choose your LLM
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)
# llm = ChatOpenAI(model="gpt-4", temperature=0)

# ✅ Setup the SQL Toolkit
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# ✅ Create the Agent
agent_executor = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,
    agent_type="openai-tools"  # or "zero-shot-react-description" for OpenAI
)

# ✅ Ask a question
response = agent_executor.invoke({"input": "Show all customers with all offers expired and expired date as well."})
#print(response)


Database tables: ['Customer', 'Offer', 'Product', 'test_table']


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


Customer, Offer, Product, test_table
Invoking: `sql_db_schema` with `{'table_names': 'Customer, Offer'}`



CREATE TABLE "Customer" (
	customer_id TEXT, 
	full_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone_number TEXT, 
	address TEXT, 
	customer_segment TEXT, 
	created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP, 
	updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP, 
	PRIMARY KEY (customer_id), 
	UNIQUE (email)
)

/*
3 rows from Customer table:
customer_id	full_name	email	phone_number	address	customer_segment	created_at	updated_at
550e8400-e29b-41d4-a716-446655440001	John Smith	john.smith@email.com	+1-555-0101	123 Main St, New York, NY 10001	Premium	2025-07-05 17:30:59	2025-07-05 17:30:59
550e8400-e29b-41d4-a716-446655440002	Sarah Johnson	sarah.johnson@email.com	+1-555-0102	456 Oak Ave, Los Angeles, CA 90210	Standard	2025-07-05 17:30:59	20

In [71]:
 
db = SQLDatabase.from_uri("sqlite:///my_database.db")
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)
    
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
agent_executor = create_sql_agent(
        llm=llm,
        toolkit=toolkit,
        verbose=True,
        agent_type="openai-tools"
    )

In [ ]:
question="show all customers with firstnames and associated offer details"
schema_context = rag_chain.invoke(f"Database schema for: {question}")
        # Create enhanced prompt with context
enhanced_question = f"""
Schema Context: {schema_context}
        
Question: {question}
        
Please use the schema context above to write the correct SQL query.
        """
        
        # Execute with SQL agent
response = agent_executor.invoke({"input": enhanced_question})
#print (response)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


Customer, Offer, Product, test_table
Invoking: `sql_db_schema` with `{'table_names': 'Product, Offer'}`
responded: The available tables are Customer, Offer, and Product.  The question asks for product names and offer details.  I will query the schema of Product and Offer tables to understand the available columns.




CREATE TABLE "Offer" (
	offer_id TEXT, 
	offer_title TEXT NOT NULL, 
	offer_description TEXT, 
	offer_type TEXT, 
	discount_percentage REAL, 
	fixed_discount_amount REAL, 
	effective_date DATE NOT NULL, 
	expiration_date DATE NOT NULL, 
	status TEXT NOT NULL, 
	customer_id TEXT, 
	product_id TEXT NOT NULL, 
	created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP, 
	updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP, 
	PRIMARY KEY (offer_id), 
	FOREIGN KEY(customer_id) REFERENCES "Customer" (customer_id) ON DELETE SET NULL, 
	FOREIGN KEY(product_id) REFERENCES "Product" (product_id) ON DELETE CASC

In [78]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Create table formatter
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

table_prompt = PromptTemplate(
    input_variables=["sql_result"],
    template="Format this SQL result as a table with column headers: {sql_result}"
)

table_formatter = table_prompt | llm | StrOutputParser()

# Use with your agent
response = agent_executor.invoke({"input": question})
formatted_table = table_formatter.invoke({"sql_result": response['output']})
print(formatted_table)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


Customer, Offer, Product, test_table
Invoking: `sql_db_schema` with `{'table_names': 'Product, Offer, Customer'}`



CREATE TABLE "Customer" (
	customer_id TEXT, 
	full_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone_number TEXT, 
	address TEXT, 
	customer_segment TEXT, 
	created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP, 
	updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP, 
	PRIMARY KEY (customer_id), 
	UNIQUE (email)
)

/*
3 rows from Customer table:
customer_id	full_name	email	phone_number	address	customer_segment	created_at	updated_at
550e8400-e29b-41d4-a716-446655440001	John Smith	john.smith@email.com	+1-555-0101	123 Main St, New York, NY 10001	Premium	2025-07-05 17:30:59	2025-07-05 17:30:59
550e8400-e29b-41d4-a716-446655440002	Sarah Johnson	sarah.johnson@email.com	+1-555-0102	456 Oak Ave, Los Angeles, CA 90210	Standard	2025-07-05 17:30:59	2025-07-05 17:30:59
550e8400-e29b-41d4-a716-446655440003	

ql\nSELECT c.customer_id, c.full_name, c.email, o.offer_description, p.product_id\nFROM CUSTOMER c\nJOIN OFFER o ON c.customer_id = o.customer_id\nJOIN PRODUCT p ON o.product_id = p.product_id;\n